In [1]:
import pandas as pd
from pathlib import Path
import json

In [5]:
with open(Path("../dane_przykladowe/veturilo.json"), "r", encoding="utf-8") as f:
    dane_json = json.load(f)
    


In [8]:
dane_df = pd.DataFrame(dane_json["network"]["stations"])
dane_df.head(3)


,id,name,latitude,longitude,timestamp,free_bikes,empty_slots,extra
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,2026-09-29T06:43:25.963728+00:00Z,2,13,"{'uid': '2585776', 'number': '9558', 'slots': ..."
1,013e8c1f7b4782baa326a337d2b00394,Marymoncka - Dewajtis,52.290173,20.950370,2026-09-29T06:43:25.958827+00:00Z,0,9,"{'uid': '2585265', 'number': '9404', 'slots': ..."
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,2026-09-29T06:43:26.194227+00:00Z,21,0,"{'uid': '4393742', 'number': '9700', 'slots': ..."


In [9]:
dane_df.dtypes

id                 str
name               str
latitude       float64
longitude      float64
timestamp          str
free_bikes       int64
empty_slots      int64
extra           object
dtype: object

In [ ]:
# wyczyszcznie kolumny z czasem
dane_df["dt"] = pd.to_datetime(dane_df["timestamp"].str[:-1])
dane_df["dt"] = dane_df["dt"].apply(lambda d: d.astimezone("Europe/Warsaw"))
dane_df = dane_df.drop(columns=['timestamp'])
dane_df

,id,name,latitude,longitude,free_bikes,empty_slots,extra,dt
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,2,13,"{'uid': '2585776', 'number': '9558', 'slots': ...",2026-09-29 08:43:25.963728+02:00
1,013e8c1f7b4782baa326a337d2b00394,Marymoncka - Dewajtis,52.290173,20.950370,0,9,"{'uid': '2585265', 'number': '9404', 'slots': ...",2026-09-29 08:43:25.958827+02:00
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,21,0,"{'uid': '4393742', 'number': '9700', 'slots': ...",2026-09-29 08:43:26.194227+02:00
3,02c9feac733886f5bdef0fe682da4b32,Darwina - Nusbauma-Hilarowicza,52.264210,21.026180,6,4,"{'uid': '2585958', 'number': '9650', 'slots': ...",2026-09-29 08:43:26.192860+02:00
4,037fadcd3c17b99a1cbc687d1d04274d,al. Jerozolimskie - Lindleya,52.226220,20.996172,15,0,"{'uid': '2585460', 'number': '9512', 'slots': ...",2026-09-29 08:43:25.962827+02:00
...,...,...,...,...,...,...,...,...
343,fda32f025901aa9743b1596d43969ab3,al. Jerozolimskie - Krucza,52.231056,21.016738,1,7,"{'uid': '2585445', 'number': '9504', 'slots': ...",2026-09-29 08:43:25.962373+02:00
344,feb31a4e8466caf11d6a16addc0b11e1,Rembielińska - Wyszogrodzka,52.288592,21.031009,18,3,"{'uid': '2585937', 'number': '9637', 'slots': ...",2026-09-29 08:43:26.192615+02:00
345,ff8fbef897530b901229c204eb46b8a1,Plac Grzybowski,52.235858,21.003731,0,12,"{'uid': '2585413', 'number': '9487', 'slots': ...",2026-09-29 08:43:25.962063+02:00
346,ff90e71b117a22c29d636bdfff8b1bbe,Sosnkowskiego - OSiR Ursus,52.186994,20.890648,2,6,"{'uid': '2585809', 'number': '9576', 'slots': ...",2026-09-29 08:43:25.964069+02:00


In [44]:
# z kolumny extra wyciągamy to co nas interesuje - w tym  bike_uids = numery przypiętych rowerów
dane_df = pd.concat(
    [
        dane_df,
        pd.json_normalize(dane_df["extra"])[["bike_uids", "number", "slots"]]
    ],
    axis=1
).drop(columns='extra')
dane_df

,id,name,latitude,longitude,free_bikes,empty_slots,dt,bike_uids,number,slots
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,2,13,2026-09-29 08:43:25.963728+02:00,"[610848, 613704]",9558,15
1,013e8c1f7b4782baa326a337d2b00394,Marymoncka - Dewajtis,52.290173,20.950370,0,9,2026-09-29 08:43:25.958827+02:00,[],9404,10
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,21,0,2026-09-29 08:43:26.194227+02:00,"[613438, 613323, 612904, 612851, 612647, 61248...",9700,14
3,02c9feac733886f5bdef0fe682da4b32,Darwina - Nusbauma-Hilarowicza,52.264210,21.026180,6,4,2026-09-29 08:43:26.192860+02:00,"[613351, 612721, 612444, 611064, 610490, 610250]",9650,10
4,037fadcd3c17b99a1cbc687d1d04274d,al. Jerozolimskie - Lindleya,52.226220,20.996172,15,0,2026-09-29 08:43:25.962827+02:00,"[613439, 612343, 612276, 612216, 611691, 61152...",9512,10
...,...,...,...,...,...,...,...,...,...,...
343,fda32f025901aa9743b1596d43969ab3,al. Jerozolimskie - Krucza,52.231056,21.016738,1,7,2026-09-29 08:43:25.962373+02:00,[613649],9504,8
344,feb31a4e8466caf11d6a16addc0b11e1,Rembielińska - Wyszogrodzka,52.288592,21.031009,18,3,2026-09-29 08:43:26.192615+02:00,"[613366, 612950, 612936, 612890, 612479, 61200...",9637,21
345,ff8fbef897530b901229c204eb46b8a1,Plac Grzybowski,52.235858,21.003731,0,12,2026-09-29 08:43:25.962063+02:00,[],9487,12
346,ff90e71b117a22c29d636bdfff8b1bbe,Sosnkowskiego - OSiR Ursus,52.186994,20.890648,2,6,2026-09-29 08:43:25.964069+02:00,"[613322, 610643]",9576,8


In [48]:
# rozdzielenie listy rowerów na nowe wiersze
dane_df = dane_df.explode("bike_uids")
dane_df

,id,name,latitude,longitude,free_bikes,empty_slots,dt,bike_uids,number,slots
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,2,13,2026-09-29 08:43:25.963728+02:00,610848,9558,15
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,2,13,2026-09-29 08:43:25.963728+02:00,613704,9558,15
1,013e8c1f7b4782baa326a337d2b00394,Marymoncka - Dewajtis,52.290173,20.950370,0,9,2026-09-29 08:43:25.958827+02:00,NaN,9404,10
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,21,0,2026-09-29 08:43:26.194227+02:00,613438,9700,14
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,21,0,2026-09-29 08:43:26.194227+02:00,613323,9700,14
...,...,...,...,...,...,...,...,...,...,...
347,fffeecd35a623278f18627b396a6fde9,Pętla Podskarbińska,52.254189,21.073601,19,0,2026-09-29 08:43:25.960072+02:00,610015,9466,5
347,fffeecd35a623278f18627b396a6fde9,Pętla Podskarbińska,52.254189,21.073601,19,0,2026-09-29 08:43:25.960072+02:00,614166,9466,5
347,fffeecd35a623278f18627b396a6fde9,Pętla Podskarbińska,52.254189,21.073601,19,0,2026-09-29 08:43:25.960072+02:00,614120,9466,5
347,fffeecd35a623278f18627b396a6fde9,Pętla Podskarbińska,52.254189,21.073601,19,0,2026-09-29 08:43:25.960072+02:00,614264,9466,5


In [54]:
# metadane stacji - niezmienne przez cały rok
dane_df[["id",
         "name",
         "latitude",
         "longitude",
         "number",
         "slots",]].drop_duplicates().dropna()


,id,name,latitude,longitude,number,slots
0,005de3bbdc044ed97ea20d32ea6401be,al. Niepodległości - Batorego,52.210833,21.007851,9558,15
1,013e8c1f7b4782baa326a337d2b00394,Marymoncka - Dewajtis,52.290173,20.950370,9404,10
2,029c42156011a06c59bc24773c914cba,Metro Centrum Nauki Kopernik,52.239358,21.031136,9700,14
3,02c9feac733886f5bdef0fe682da4b32,Darwina - Nusbauma-Hilarowicza,52.264210,21.026180,9650,10
4,037fadcd3c17b99a1cbc687d1d04274d,al. Jerozolimskie - Lindleya,52.226220,20.996172,9512,10
...,...,...,...,...,...,...
343,fda32f025901aa9743b1596d43969ab3,al. Jerozolimskie - Krucza,52.231056,21.016738,9504,8
344,feb31a4e8466caf11d6a16addc0b11e1,Rembielińska - Wyszogrodzka,52.288592,21.031009,9637,21
345,ff8fbef897530b901229c204eb46b8a1,Plac Grzybowski,52.235858,21.003731,9487,12
346,ff90e71b117a22c29d636bdfff8b1bbe,Sosnkowskiego - OSiR Ursus,52.186994,20.890648,9576,8


In [55]:
# stan stacji w danej chwili
dane_df[
    [
        "id",
        "dt",
        "free_bikes",
        "empty_slots"
    ]
].drop_duplicates().dropna()


,id,dt,free_bikes,empty_slots
0,005de3bbdc044ed97ea20d32ea6401be,2026-09-29 08:43:25.963728+02:00,2,13
1,013e8c1f7b4782baa326a337d2b00394,2026-09-29 08:43:25.958827+02:00,0,9
2,029c42156011a06c59bc24773c914cba,2026-09-29 08:43:26.194227+02:00,21,0
3,02c9feac733886f5bdef0fe682da4b32,2026-09-29 08:43:26.192860+02:00,6,4
4,037fadcd3c17b99a1cbc687d1d04274d,2026-09-29 08:43:25.962827+02:00,15,0
...,...,...,...,...
343,fda32f025901aa9743b1596d43969ab3,2026-09-29 08:43:25.962373+02:00,1,7
344,feb31a4e8466caf11d6a16addc0b11e1,2026-09-29 08:43:26.192615+02:00,18,3
345,ff8fbef897530b901229c204eb46b8a1,2026-09-29 08:43:25.962063+02:00,0,12
346,ff90e71b117a22c29d636bdfff8b1bbe,2026-09-29 08:43:25.964069+02:00,2,6


In [56]:
# stan rowerów w danej chwili - gdzie jest który przypięty
dane_df[["bike_uids",  "dt", "id"]].drop_duplicates().dropna()


,bike_uids,dt,id
0,610848,2026-09-29 08:43:25.963728+02:00,005de3bbdc044ed97ea20d32ea6401be
0,613704,2026-09-29 08:43:25.963728+02:00,005de3bbdc044ed97ea20d32ea6401be
2,613438,2026-09-29 08:43:26.194227+02:00,029c42156011a06c59bc24773c914cba
2,613323,2026-09-29 08:43:26.194227+02:00,029c42156011a06c59bc24773c914cba
2,612904,2026-09-29 08:43:26.194227+02:00,029c42156011a06c59bc24773c914cba
...,...,...,...
347,610015,2026-09-29 08:43:25.960072+02:00,fffeecd35a623278f18627b396a6fde9
347,614166,2026-09-29 08:43:25.960072+02:00,fffeecd35a623278f18627b396a6fde9
347,614120,2026-09-29 08:43:25.960072+02:00,fffeecd35a623278f18627b396a6fde9
347,614264,2026-09-29 08:43:25.960072+02:00,fffeecd35a623278f18627b396a6fde9


In [57]:
df = pd.DataFrame({"Dzień": [1, 2, 3, 4, 5], "Wartość": [10, 25, 15, 30, 45]})
df

,Dzień,Wartość
0,1,10
1,2,25
2,3,15
3,4,30
4,5,45


In [58]:
# delta między jednym dniem a drugim
df["Wartość"].pct_change()

0    NaN
1    1.5
2   -0.4
3    1.0
4    0.5
Name: Wartość, dtype: float64

In [ ]:
# LAG - przesunięcie o 1 wiersz w dół
df["Wartość"].shift(1)

0     NaN
1    10.0
2    25.0
3    15.0
4    30.0
Name: Wartość, dtype: float64

In [61]:
# LEAD - przesunięcie w górę
df["Wartość"].shift(-2)


0    15.0
1    30.0
2    45.0
3     NaN
4     NaN
Name: Wartość, dtype: float64

In [62]:
df['lead_2'] = df["Wartość"].shift(-2)
df["lag_1"] = df["Wartość"].shift(1)
df


,Dzień,Wartość,lead_2,lag_1
0,1,10,15.0,NaN
1,2,25,30.0,10.0
2,3,15,45.0,25.0
3,4,30,NaN,15.0
4,5,45,NaN,30.0


In [ ]:
# suma skumulowana
df["Wartość"].cumsum()

0     10
1     35
2     50
3     80
4    125
Name: Wartość, dtype: int64

In [66]:
df["Wartość"].cummax()


0    10
1    25
2    25
3    30
4    45
Name: Wartość, dtype: int64

In [67]:
df["Wartość"].cumprod()


0         10
1        250
2       3750
3     112500
4    5062500
Name: Wartość, dtype: int64